# Notebook 03 — Feature Engineering
### NER System | Sports & Political News

Convert labeled data to IOB format and engineer token-level features for CRF model.

In [1]:
import pandas as pd
import numpy as np
import json, os, re, warnings
warnings.filterwarnings('ignore')
print('Libraries loaded OK')

Libraries loaded OK


## Step 1 — Load Cleaned Data

In [2]:
df = pd.read_csv('../data/cleaned_data.csv')
print(f'Loaded: {len(df)} rows')
df.head(3)

Loaded: 1737 rows


,article_id,domain,sentence,person_entities,location_entities,has_person,has_location,has_entity,word_count
0,1,Sports,Max Verstappen has threatened to quit Formula ...,Max Verstappen,NONE,True,False,True,26
1,1,Sports,Governing body the FIA said two weeks ago that...,NONE,NONE,False,False,False,33
2,1,Sports,But opposition has emerged in subsequent talks...,NONE,NONE,False,False,False,30


## Step 2 — Convert to IOB Format

IOB (Inside-Outside-Beginning) format tags each token:
- `B-PERSON` = Beginning of person entity
- `I-PERSON` = Inside person entity
- `B-LOCATION` = Beginning of location entity
- `I-LOCATION` = Inside location entity
- `O` = Outside (not an entity)

In [3]:
def tokenize(sentence):
    return str(sentence).split()

def get_iob_tags(sentence, person_str, location_str):
    tokens = tokenize(sentence)
    tags   = ['O'] * len(tokens)

    # Parse entities
    persons   = [p.strip() for p in str(person_str).split(',') if p.strip() and p.strip()!='NONE']
    locations = [l.strip() for l in str(location_str).split(',') if l.strip() and l.strip()!='NONE']

    sentence_lower = sentence.lower()

    def tag_entity(tokens, tags, entity, label):
        ent_tokens = entity.split()
        n = len(ent_tokens)
        for i in range(len(tokens) - n + 1):
            if [t.lower().strip('.,!?;:"') for t in tokens[i:i+n]] == [e.lower() for e in ent_tokens]:
                tags[i] = f'B-{label}'
                for j in range(1, n):
                    tags[i+j] = f'I-{label}'
        return tags

    for p in persons:
        tags = tag_entity(tokens, tags, p, 'PERSON')
    for l in locations:
        tags = tag_entity(tokens, tags, l, 'LOCATION')

    return tokens, tags

# Test it
tokens, tags = get_iob_tags('Max Verstappen raced in Monaco today.', 'Max Verstappen', 'Monaco')
print('Sample IOB output:')
for t, tag in zip(tokens, tags):
    print(f'  {t:20s} → {tag}')

Sample IOB output:
  Max                  → B-PERSON
  Verstappen           → I-PERSON
  raced                → O
  in                   → O
  Monaco               → B-LOCATION
  today.               → O


## Step 3 — Build Full IOB Dataset

In [4]:
iob_data = []
skipped  = 0

for _, row in df.iterrows():
    try:
        tokens, tags = get_iob_tags(
            str(row['sentence']),
            str(row['person_entities']),
            str(row['location_entities'])
        )
        if len(tokens) > 0:
            iob_data.append({
                'article_id': row['article_id'],
                'domain'    : row['domain'],
                'sentence'  : str(row['sentence']),
                'tokens'    : tokens,
                'tags'      : tags
            })
    except Exception as e:
        skipped += 1

print(f'IOB records created : {len(iob_data)}')
print(f'Skipped             : {skipped}')

# Count tag distribution
from collections import Counter
all_tags = [t for d in iob_data for t in d['tags']]
tag_counts = Counter(all_tags)
print('\n=== TAG DISTRIBUTION ===')
for tag, count in sorted(tag_counts.items()):
    print(f'  {tag:15s}: {count}')

IOB records created : 1737
Skipped             : 0

=== TAG DISTRIBUTION ===
  B-LOCATION     : 673
  B-PERSON       : 734
  I-LOCATION     : 112
  I-PERSON       : 426
  O              : 35649


## Step 4 — CRF Feature Engineering

For each token we extract features the CRF model will use to learn entity boundaries.

In [5]:
def word_features(tokens, i):
    word = tokens[i]
    features = {
        'word.lower'      : word.lower(),
        'word.isupper'    : word.isupper(),
        'word.istitle'    : word.istitle(),
        'word.isdigit'    : word.isdigit(),
        'word.prefix2'    : word[:2].lower(),
        'word.prefix3'    : word[:3].lower(),
        'word.suffix2'    : word[-2:].lower(),
        'word.suffix3'    : word[-3:].lower(),
        'word.has_hyphen' : '-' in word,
        'word.has_digit'  : any(c.isdigit() for c in word),
        'word.length'     : len(word),
        'BOS'             : i == 0,
        'EOS'             : i == len(tokens) - 1,
    }
    if i > 0:
        prev = tokens[i-1]
        features.update({
            'prev_word.lower'  : prev.lower(),
            'prev_word.istitle': prev.istitle(),
            'prev_word.isupper': prev.isupper(),
        })
    else:
        features['BOS'] = True

    if i < len(tokens) - 1:
        nxt = tokens[i+1]
        features.update({
            'next_word.lower'  : nxt.lower(),
            'next_word.istitle': nxt.istitle(),
            'next_word.isupper': nxt.isupper(),
        })
    else:
        features['EOS'] = True

    return features

def sent_features(tokens):
    return [word_features(tokens, i) for i in range(len(tokens))]

def sent_labels(tags):
    return tags

# Test
sample = iob_data[0]
feats  = sent_features(sample['tokens'])
print(f'Sample token: "{sample["tokens"][0]}"')
print(f'Features:')
for k, v in feats[0].items():
    print(f'  {k:22s}: {v}')

Sample token: "Max"
Features:
  word.lower            : max
  word.isupper          : False
  word.istitle          : True
  word.isdigit          : False
  word.prefix2          : ma
  word.prefix3          : max
  word.suffix2          : ax
  word.suffix3          : max
  word.has_hyphen       : False
  word.has_digit        : False
  word.length           : 3
  BOS                   : True
  EOS                   : False
  next_word.lower       : verstappen
  next_word.istitle     : True
  next_word.isupper     : False


## Step 5 — Prepare SpaCy Training Format

In [6]:
def build_spacy_data(iob_records):
    spacy_data = []
    for rec in iob_records:
        sentence = rec['sentence']
        tokens   = rec['tokens']
        tags     = rec['tags']
        entities = []
        i = 0
        while i < len(tags):
            if tags[i].startswith('B-'):
                label = tags[i][2:]
                start = len(' '.join(tokens[:i]))
                if i > 0: start += 1
                j = i + 1
                while j < len(tags) and tags[j].startswith('I-'):
                    j += 1
                end = len(' '.join(tokens[:j]))
                if i > 0: end += 1
                # Verify span matches
                span_text = ' '.join(tokens[i:j])
                if span_text in sentence:
                    char_start = sentence.find(span_text)
                    char_end   = char_start + len(span_text)
                    entities.append((char_start, char_end, label))
                i = j
            else:
                i += 1
        spacy_data.append((sentence, {'entities': entities}))
    return spacy_data

spacy_data = build_spacy_data(iob_data)
print(f'SpaCy training records: {len(spacy_data)}')
# Show sample
for text, annot in spacy_data[:3]:
    if annot['entities']:
        print(f'  Text: {text[:60]}...')
        print(f'  Entities: {annot["entities"]}')
        break

SpaCy training records: 1737
  Text: Max Verstappen has threatened to quit Formula 1 at the end o...
  Entities: [(0, 14, 'PERSON')]


## Step 6 — Save Feature Data

In [7]:
os.makedirs('../data', exist_ok=True)

# Save IOB data as JSON
with open('../data/iob_data.json', 'w') as f:
    json.dump(iob_data, f, indent=2)
print(f'Saved: ../data/iob_data.json ({len(iob_data)} records)')

# Save SpaCy training data as JSON
spacy_save = [{'text': t, 'entities': [[e[0],e[1],e[2]] for e in a['entities']]} for t,a in spacy_data]
with open('../data/spacy_training_data.json', 'w') as f:
    json.dump(spacy_save, f, indent=2)
print(f'Saved: ../data/spacy_training_data.json ({len(spacy_save)} records)')

# Stats
with_ents = sum(1 for t,a in spacy_data if a['entities'])
print(f'\nRecords with entities : {with_ents}')
print(f'Records without       : {len(spacy_data)-with_ents}')

Saved: ../data/iob_data.json (1737 records)
Saved: ../data/spacy_training_data.json (1737 records)

Records with entities : 825
Records without       : 912


---
## Notebook 03 Complete ✅
**Next → Notebook 04: Model Training**